# Dynamics 05 — Agents-programmes à budget explicite : compagnon C#

**Navigation** : [<< Dynamics-04-Open-Source-Game-Theory-Python](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) | [Index](README.md) | [companion Lean](Dynamics-05-Agents-Bornes-Lean.ipynb) | [Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python >>](Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python.ipynb)

Ce notebook est le compagnon C# du module Lean 4 [`ProgramGames.Bounded`](../game_theory_lean/ProgramGames/Bounded.lean) du lake [`game_theory_lean`](../game_theory_lean/README.md). Le module represente explicitement le **code public** et le **budget de raisonnement fini** d'un agent-programme (modele structurel de Barasz et al. 2014 et Critch 2016), avec un interprete **total** : un budget nul produit immediatement une action, aucun resultat ne depend d'une recherche de preuve non bornee.

Le pivot conceptuel general reste [`Dynamics-04-Open-Source-Game-Theory-Python.ipynb`](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) ; ce compagnon se concentre sur le modele borne et ne duplique pas son contenu.

Source primaire : [notebook Python versionné](Dynamics-05-Agents-Bornes-Python.ipynb). Les références bibliographiques sont héritées ; aucun processus Lean n’est exécuté.

**Version jumelle :** [Rust](Dynamics-05-Agents-Bornes-Rust.ipynb).


## Ce que fait ce notebook - et ce qu'il ne fait pas

Aucune invocation Lean ne se produit ici : les noms et énoncés Lean sont des références. Le role du compagnon Python est double :

1. **reproduire independamment** la famille finie de bots (`cooperateBot`, `defectBotBounded`, `mirrorBot`, `basicFamily`, `canonicalPD`) depuis zero, sans importer le code Lean ;
2. **rejouer les certificats calculables** du module et comparer chaque verdict C# au theoreme Lean correspondant.

La distinction est fondamentale : le C# vérifie une **matrice finie**. Dans le module cité, certains énoncés portent sur des cas fixés ; `defectBotBounded_unexploitable` quantifie les listes d’opposants, et `programNashCheck_eq_true` fournit séparément une équivalence générale. Un verdict C# conforme ne remplace jamais la preuve formelle - il la rend **inspectable** : chaque ligne du tableau ci-dessous peut etre relue, modifiee, contredite par l'etudiant. Ce notebook ne formalise ni logique de prouvabilite ni theoreme de Lob, et n'extrapole pas vers Godel : le module Lean lui-meme s'en garde explicitement.


## Le code public et l'agent borne

Un `BoundedAgent` porte un code public inspectable (un des trois bots temoins) et un budget de raisonnement naturel. La traduction Python est fidele a la structure Lean `ProgramCode` / `BoundedAgent` :

Le port nomme ses équivalents `Act`, `Outcome`, `Mutual`, `Unexploitable` et `ProgramNash`.


In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Code public et budget suffisent : aucune mémoire ni simulation récurrente.
public enum ProgramCode
{
    CooperateBot,
    DefectBot,
    Mirror
}

public record BoundedAgent(ProgramCode Code, int Budget);
var cooperateBot = new BoundedAgent(ProgramCode.CooperateBot, 0);
var defectBot = new BoundedAgent(ProgramCode.DefectBot, 0);
var mirrorBot = new BoundedAgent(ProgramCode.Mirror, 1);
var basicFamily = new List<BoundedAgent>
{
    cooperateBot,
    defectBot,
    mirrorBot
};
Console.WriteLine("Famille témoin : " + string.Join(", ", basicFamily));


Famille témoin : BoundedAgent { Code = CooperateBot, Budget = 0 }, BoundedAgent { Code = DefectBot, Budget = 0 }, BoundedAgent { Code = Mirror, Budget = 1 }


**Lecture de la sortie — la famille temoin et ses budgets.** Trois lignes suffisent a poser le decor : `BoundedAgent { Code = CooperateBot, Budget = 0 }`, `BoundedAgent { Code = DefectBot, Budget = 0 }`, `BoundedAgent { Code = Mirror, Budget = 1 }`. Les deux bots vivent a budget nul — leur decision ne regarde ni l'historique ni l'adversaire, aucune memoire a payer. Le miroir seul porte `budget=1` : inspecter le code adverse avec un budget positif est le test représenté, et c'est ce cout qui distinguera le miroir riche (b=1, il reflete) du miroir casse (b=0, l'exercice 1 y viendra) — la meme ligne de code, deux comportements selon un seul entier. Toute la question du notebook est deja la : que reste-t-il du jeu quand la lecture a un prix ?


## L'interprete total `act`

L'interprete ne fait aucune recherche de preuve : il inspecte le code adverse et le budget, puis produit une action. A **budget nul**, `mirror` deve (la règle à budget nul prescrit directement la défection) ; a **budget positif**, il coopere sauf contre le code explicitement defecteur :


In [2]:
// L'interprète total effectue une inspection du code et du budget.
string Act(BoundedAgent agent, ProgramCode opponent)
{
    if (agent.Code == ProgramCode.CooperateBot)
        return "cooperate";
    if (agent.Code == ProgramCode.DefectBot)
        return "defect";
    if (agent.Budget == 0)
        return "defect";
    return opponent == ProgramCode.DefectBot ? "defect" : "cooperate";
}

(string, string) Outcome(BoundedAgent a, BoundedAgent b) => (Act(a, b.Code), Act(b, a.Code));
foreach (var a in basicFamily)
    foreach (var b in basicFamily)
        Console.WriteLine($"{a} / {b} = {Outcome(a, b)}");


BoundedAgent { Code = CooperateBot, Budget = 0 } / BoundedAgent { Code = CooperateBot, Budget = 0 } = (cooperate, cooperate)


BoundedAgent { Code = CooperateBot, Budget = 0 } / BoundedAgent { Code = DefectBot, Budget = 0 } = (cooperate, defect)


BoundedAgent { Code = CooperateBot, Budget = 0 } / BoundedAgent { Code = Mirror, Budget = 1 } = (cooperate, cooperate)


BoundedAgent { Code = DefectBot, Budget = 0 } / BoundedAgent { Code = CooperateBot, Budget = 0 } = (defect, cooperate)


BoundedAgent { Code = DefectBot, Budget = 0 } / BoundedAgent { Code = DefectBot, Budget = 0 } = (defect, defect)


BoundedAgent { Code = DefectBot, Budget = 0 } / BoundedAgent { Code = Mirror, Budget = 1 } = (defect, defect)


BoundedAgent { Code = Mirror, Budget = 1 } / BoundedAgent { Code = CooperateBot, Budget = 0 } = (cooperate, cooperate)


BoundedAgent { Code = Mirror, Budget = 1 } / BoundedAgent { Code = DefectBot, Budget = 0 } = (defect, defect)


BoundedAgent { Code = Mirror, Budget = 1 } / BoundedAgent { Code = Mirror, Budget = 1 } = (cooperate, cooperate)


**Lecture chiffree — les issues bornees, ligne par ligne.** La sortie egraine `outcome_bounded` sur chaque paire. Les lignes COOPERATE_BOT : `(cooperate, cooperate)` contre son semblable, `(cooperate, defect)` contre DEFECT_BOT — le coopereur ne s'adapte pas, il subit. Les lignes DEFECT_BOT font le symetrique : `(defect, cooperate)` puis `(defect, defect)`. La ligne qui instruit est la troisieme colonne : contre `MIRROR b=1`, COOPERATE_BOT obtient `('cooperate', 'cooperate')` — le miroir inspecte une fois le code du coopérateur et coopère ; DEFECT_BOT contre le miroir obtient la paire symetrique de defection : le miroir détecte le code défecteur et dévie. Deux bots a budget nul ont des issues independantes du budget ; le miroir a b=1 est le seul dont l'issue depende de ce qu'il a pu lire — la famille temoin deliberate ce contraste.


La matrice des issues de la famille temoin se lit d'un coup d'oeil :


In [3]:
// Les neuf issues ordonnées gardent l'action ligne puis colonne.
Console.WriteLine("Matrice des issues (C / D)");
foreach (var row in basicFamily)
{
    var values = basicFamily.Select(col => Outcome(row, col));
    Console.WriteLine($"{row.Code} : " + string.Join(" ", values.Select(v => $"{char.ToUpperInvariant(v.Item1[0])}/{char.ToUpperInvariant(v.Item2[0])}")));
}


Matrice des issues (C / D)


CooperateBot : C/C C/D C/C


DefectBot : D/C D/D D/D


Mirror : C/C D/D C/C


**Lecture de la matrice — trois lignes, trois caracteres.** Le tableau croise condense la sortie precedente : ligne = agent, colonne = adversaire, format `action de l'agent / action de l'adversaire`. La ligne COOPERATE_BOT affiche trois `C/` : il coopere contre tous, sans exception ni memoire. La ligne DEFECT_BOT affiche trois `D/` : defection systématique, le complement exact. La ligne MIRROR dépend du code public de son adversaire : `C/C` face au coopereur, `D/D` face au defecteur, `C/C` face a lui-meme — ce résultat provient du test sur les trois codes, sans historique ni exécution itérée. Cette coïncidence des issues est limitée à cette famille finie.


## Le dilemme du prisonnier canonique et le rang de paiement

Le parametrage canonique est `T=5, R=3, P=1, S=0`. Le module definit un **rang de paiement fini** `payoffRank` (`CC=3, CD=0, DC=5, DD=1`) et prouve (`payoffRank_le_iff`) que ce rang preserve exactement l'ordre des paiements du jeu canonique - evitant de comparer des reels arbitraires :


In [4]:
// Les valeurs du jeu canonique et le rang fini coïncident sur les quatre issues.
int Pay(string row, string col)
{
    if (row == "cooperate")
        return col == "cooperate" ? 3 : 0;
    return col == "cooperate" ? 5 : 1;
}

int Rank(string row, string col) => (row, col) switch
{
    ("cooperate", "cooperate") => 3,
    ("cooperate", "defect") => 0,
    ("defect", "cooperate") => 5,
    _ => 1
};
foreach (var row in new[]
{
    "cooperate",
    "defect"
}

)
    foreach (var col in new[]
    {
        "cooperate",
        "defect"
    }

    )
        Console.WriteLine($"('{row}', '{col}') -> stage_payoff={Pay(row, col)}, payoffRank={Rank(row, col)}");


('cooperate', 'cooperate') -> stage_payoff=3, payoffRank=3


('cooperate', 'defect') -> stage_payoff=0, payoffRank=0


('defect', 'cooperate') -> stage_payoff=5, payoffRank=5


('defect', 'defect') -> stage_payoff=1, payoffRank=1


**Lecture chiffree — le rang de paiement encode l'ordre du dilemme.** La sortie applique le PD canonique `{'T': 5, 'R': 3, 'P': 1, 'S': 0}` aux quatre issues. Chaque ligne porte deux nombres : `('cooperate','cooperate') -> stage_payoff=3, payoffRank=3`, `('cooperate','defect') -> 0, 0`, `('defect','cooperate') -> 5, 5`, `('defect','defect') -> 1, 1`. Ici rang et gain coïncident chiffre à chiffre parce que `payoffRank` reprend explicitement les quatre valeurs canoniques, pas simplement parce qu'elles sont distinctes. Il préserve l'ordre T > R > P > S pour ce jeu ; cette vérification n'établit aucune comparabilité cardinale entre jeux différents. Le certificat final du notebook (16 paires) portera precisement sur ce rang.


## Les organes calculables

Le module expose trois organes booléens — `mutualCooperationCheck`, `unexploitableCheck` et `programNashCheck` — correspondant aux définitions propositionnelles. L’équivalence générale `programNashCheck_eq_true` est explicitement prouvée dans Lean pour le PD canonique. Le calcul ci-dessous reproduit ces organes sur les familles finies du carnet ; aucune invocation Lean n’a lieu ici.


In [5]:
// L'inexploitation interdit uniquement la coopération unilatérale C/D.
bool Mutual(BoundedAgent a, BoundedAgent b) => Outcome(a, b) == ("cooperate", "cooperate");
bool Unexploitable(BoundedAgent a, IEnumerable<BoundedAgent> opponents) => opponents.All(b => Outcome(a, b) != ("cooperate", "defect"));
bool ProgramNash(IEnumerable<BoundedAgent> family, BoundedAgent left, BoundedAgent right)
{
    var current = Outcome(left, right);
    return family.All(alt =>
    {
        var x = Outcome(alt, right);
        var y = Outcome(left, alt);
        return Rank(x.Item1, x.Item2) <= Rank(current.Item1, current.Item2) && Rank(y.Item2, y.Item1) <= Rank(current.Item2, current.Item1);
    });
}

Console.WriteLine("Organes finis : coopération, absence de C/D et équilibre relatif prêts.");


Organes finis : coopération, absence de C/D et équilibre relatif prêts.


## Exercice 1 - Le miroir sans budget

Le module documente qu'a budget nul, `mirror` deve. On construit `mirror_broke = BoundedAgent(ProgramCode.MIRROR, 0)`. **Question** : ce miroir sans budget est-il inexploitable dans `basic_family` ? La cooperation mutuelle est-elle encore possible ? Completer la fonction pour calculer les deux verdicts, puis verifier contre l'intuition : un miroir a budget nul se comporte structurellement comme quel bot de la famille ?

Dans ce port, compléter le délégué C# correspondant ; les sondes utilisent aussi des familles variantes.


In [6]:
// Exercice ouvert : deux verdicts pour un miroir de budget donné.
// Utiliser l’inexploitation dans la famille et la coopération avec soi-même.
// Comparer le miroir sans budget au défecteur.
var mirrorBroke = new BoundedAgent(ProgramCode.Mirror, 0);
Func<BoundedAgent, List<BoundedAgent>, (bool, bool)?> exercice1 = (agent, family) => null;
var cases1 = new[]
{
    (mirrorBroke, basicFamily),
    (mirrorBot, basicFamily),
    (cooperateBot, new List<BoundedAgent> { defectBot })
};
var expected1 = cases1.Select(c => (Unexploitable(c.Item1, c.Item2), Mutual(c.Item1, c.Item1))).ToArray();
var replies1 = cases1.Select(c => exercice1(c.Item1, new List<BoundedAgent>(c.Item2))).ToArray();
if (replies1.All(r => r == null))
    Console.WriteLine("Exercice 1 à compléter.");
else
{
    if (replies1.Any(r => r == null))
        throw new Exception("Réponse partielle");
    for (int i = 0; i < cases1.Length; i++)
        if (replies1[i].Value != expected1[i])
            throw new Exception("Verdict incorrect");
    Console.WriteLine("Exercice 1 : sondes réussies.");
}


Exercice 1 à compléter.


## Exercice 2 - Le paiement du miroir contre le defecteur

Le theoreme `mirror_basicFamily_unexploitable` affirme que le miroir a budget positif est inexploitable dans la famille temoin : il deve contre `defectBotBounded`, cooperant avec les autres. **Question** : quel paiement le miroir obtient-il effectivement contre chaque membre de la famille, et quel membre est son pire adversaire ? Completer le calcul du dict des paiements et du pire adversaire, puis conclure : inexploitable signifie-t-il optimal ?

Dans ce port, compléter le délégué C# correspondant ; les sondes utilisent aussi des familles variantes.


In [7]:
// Exercice ouvert : paiements contre chaque adversaire et un pire adversaire.
// Calculer le gain du miroir contre chaque membre dans le bon ordre.
// Trouver un pire adversaire et distinguer protection et optimalité.
Func<BoundedAgent, List<BoundedAgent>, (Dictionary<BoundedAgent, int>, BoundedAgent)?> exercice2 = (agent, family) => null;
var cases2 = new[]
{
    (mirrorBot, basicFamily),
    (mirrorBroke, basicFamily),
    (cooperateBot, new List<BoundedAgent> { defectBot })
};
var expected2 = cases2.Select(c => c.Item2.ToDictionary(o => o, o =>
{
    var result = Outcome(c.Item1, o);
    return Pay(result.Item1, result.Item2);
})).ToArray();
var replies2 = cases2.Select(c => exercice2(c.Item1, new List<BoundedAgent>(c.Item2))).ToArray();
if (replies2.All(r => r == null))
    Console.WriteLine("Exercice 2 à compléter.");
else
{
    if (replies2.Any(r => r == null))
        throw new Exception("Réponse partielle");
    for (int i = 0; i < cases2.Length; i++)
    {
        var(gains, worst) = replies2[i].Value;
        if (gains == null
        || worst == null
        || gains.Count != expected2[i].Count
        || expected2[i].Any(k => !gains.ContainsKey(k.Key)
        || gains[k.Key] != k.Value)
        || !expected2[i].ContainsKey(worst)
        || expected2[i][worst] != expected2[i].Values.Min())
            throw new Exception("Paiement incorrect");
    }

    Console.WriteLine("Exercice 2 : sondes réussies.");
}


Exercice 2 à compléter.


## Rejeu des certificats Lean

Les huit lignes ci-dessous confrontent des calculs finis à des énoncés Lean cités : plusieurs sont des cas fixés de la famille témoin, et la colonne rappelle leur référence. `defectBotBounded_unexploitable` quantifie les listes d’opposants ; `payoffRank_le_iff` compare les 16 couples d’issues du seul `canonicalPD`. Les lignes 6 et 7 calculent le même profil, sous forme propositionnelle puis booléenne. L’équivalence générale supplémentaire `programNashCheck_eq_true` est distincte de ces huit résultats. Aucun Lean n’est exécuté ici.


**Protocole de lecture — decoder une ligne de certificat.** Chaque ligne de sortie a suivre a la meme grammaire : un nom court (`mirror_mirror`), un verdict `C#=True`, une fleche `<-`, puis l'enonce Lean dont c'est la replique (`MutualCooperationBounded mirrorBot mirrorBot`). Trois choses a ne pas confondre : le verdict `True` dit que LE CALCUL C# rend vrai sur la famille temoin, pas que l'énoncé Lean cité a été vérifié par ce carnet ; le nom court est une cle de lecture, pas l'enonce ; et la fleche pointe de la preuve calculee vers la cible formelle — le sens du rejeu. Les enonces quantifies (sur toute une famille d'adversaires) se lisent differemment des egalites simples : l'interpretation qui suit cette section detaille ce partage.


In [8]:
// Les noms Lean sont des références : aucun processus Lean n'est invoqué.
var certificates = new List<(string name, bool value, string lean)>();
void Verify(string name, bool value, string lean)
{
    certificates.Add((name, value, lean));
    Console.WriteLine($"{name} C#={value} <- {lean}");
}

Verify("cooperate_cooperate", Mutual(cooperateBot, cooperateBot), "MutualCooperationBounded cooperateBot cooperateBot");
Verify("defect_defect", Outcome(defectBot, defectBot) == ("defect", "defect"), "outcomeBounded defectBotBounded defectBotBounded = (defect, defect)");


cooperate_cooperate C#=True <- MutualCooperationBounded cooperateBot cooperateBot


defect_defect C#=True <- outcomeBounded defectBotBounded defectBotBounded = (defect, defect)


**Lecture chiffree — les deux premieres egalites calculees.** `cooperate_cooperate C#=True <- MutualCooperationBounded cooperateBot cooperateBot` : les deux coopereurs bornes cooperent mutuellement — une egalite que la matrice de la section precedente montrait deja case par case (`C/ C` en haut a gauche), ici confrontee a l'enonce exact. `defect_defect C#=True <- outcomeBounded defectBotBounded defectBotBounded = (defect, defect)` : la paire de defecteurs se defie mutuellement, meme egalite sur l'autre coin diagonal. Les deux certificats bornent la matrice par ses deux extremites pures — cooperation universelle d'un cote, defection universelle de l'autre — avant que les suivants n'attaquent les cas ou la strategie depend de l'autre.


### Ce que chaque certificat engage

Les deux premiers certificats sont des **egalites calculees** sur la matrice finie : on evalue une decision et on la compare au membre droit de l'enonce Lean. Les suivants melangent deux natures : des egalites du meme genre, et un enonce **quantifie** (`UnexploitableInFamily defectBotBounded opponents`), dont le verdict ne se lit plus cellule par cellule mais sur **toute** la famille d'adversaires.

Un calcul Python peut *corroborer* un enonce quantifie sur la famille temoin ; il ne peut pas l'**etablir** pour une famille arbitraire. C'est le partage des roles que ce rejeu met en scene : le calcul exhibe un contre-exemple s'il en existe un, la preuve Lean fournit la quantification.

In [9]:
// Trois certificats : miroir et absence de coopération unilatérale dans la famille.
Verify("mirror_mirror", Mutual(mirrorBot, mirrorBot), "MutualCooperationBounded mirrorBot mirrorBot");
Verify("defectBotBounded_unexploitable", Unexploitable(defectBot, basicFamily), "UnexploitableInFamily defectBotBounded opponents");
Verify("mirror_basicFamily_unexploitable", Unexploitable(mirrorBot, basicFamily), "unexploitableCheck mirrorBot basicFamily = true");


mirror_mirror C#=True <- MutualCooperationBounded mirrorBot mirrorBot


defectBotBounded_unexploitable C#=True <- UnexploitableInFamily defectBotBounded opponents


mirror_basicFamily_unexploitable C#=True <- unexploitableCheck mirrorBot basicFamily = true


**Lecture chiffree — miroir, puis les deux inexploitabilites.** Trois lignes, trois natures. `mirror_mirror C#=True <- MutualCooperationBounded mirrorBot mirrorBot` : deux miroirs a budget positif cooperent entre eux — chacun inspecte une fois le code MIRROR et choisit coopération. `defectBotBounded_unexploitable C#=True <- UnexploitableInFamily defectBotBounded opponents` : le defecteur est inexploitable DANS la famille — le couple (cooperate, defect) ne se produit jamais pour lui, et l'énoncé Lean cité quantifie les listes d'opposants, tandis que ce carnet teste la famille témoin. `mirror_basicFamily_unexploitable C#=True <- unexploitableCheck mirrorBot basicFamily` : le miroir aussi, sur la meme famille temoin. Noter la difference d'echelle des trois preuves : la premiere se verifie sur une case, les deux autres exigent le balayage de la famille entiere.


### La proposition et son miroir decidable

Les certificats 6 et 7 présentent le même profil fixé sous deux formes : la proposition d’équilibre et la décision booléenne. Le module comporte aussi `programNashCheck_eq_true`, une équivalence générale supplémentaire entre la décision et la proposition ; elle ne transforme pas ces deux lignes calculées en preuves universelles.

Les deux formes ne sont pas redondantes : une proposition quantifiee se *prouve*, un organe booleen s'*execute*. Confronter l'une a l'autre est ce qui rend une divergence visible.

In [10]:
// Les seize comparaisons croisent les quatre issues et leurs rangs.
Verify("defect_profile_programNash", ProgramNash(basicFamily, defectBot, defectBot), "ProgramNashBounded canonicalPD basicFamily defectBotBounded defectBotBounded");
Verify("defect_profile_check", ProgramNash(basicFamily, defectBot, defectBot), "programNashCheck basicFamily defectBotBounded defectBotBounded = true");
var actions = new[]
{
    "cooperate",
    "defect"
};
var pairs = (
    from a in actions
    from b in actions
    select (a, b)).ToArray();
bool rankOk = pairs.All(a => pairs.All(b => (Rank(a.a, a.b) <= Rank(b.a, b.b)) == (Pay(a.a, a.b) <= Pay(b.a, b.b))));
Verify("payoffRank_le_iff (16 paires)", rankOk, "payoffRank_le_iff : rang fini iff ordre des paiements");
Console.WriteLine($"{certificates.Count(c => c.value)}/{certificates.Count} certificats conformes");
if (certificates.Count != 8 || certificates.Any(c => !c.value))
    throw new Exception("Certificat fini divergent");


defect_profile_programNash C#=True <- ProgramNashBounded canonicalPD basicFamily defectBotBounded defectBotBounded


defect_profile_check C#=True <- programNashCheck basicFamily defectBotBounded defectBotBounded = true


payoffRank_le_iff (16 paires) C#=True <- payoffRank_le_iff : rang fini iff ordre des paiements


8/8 certificats conformes


**Lecture chiffree — la ligne la plus riche : les 16 paires.** Apres les certificats 6 et 7 (l'equilibre relatif de la defection mutuelle, en double ecriture proposition/organe), la sortie affiche `payoffRank_le_iff (16 paires) C#=True <- payoffRank_le_iff : rang fini iff ordre des paiements`. Le compte 16 est la vraie information : toutes les paires de paiements du PD canonique (issues et rangs croises) sont passees en revue, et pour chacune l'equivalence a ete evaluee — le rang et l'ordre s'accordent paire par paire, sans exception. La ligne finale `8/8 certificats conformes` solde le rejeu : les huit vérifications annoncées ont chacune leur résultat calculé en C#, le partage des roles reste ce que la section suivante en dit.


Les 8 verdicts C# sont conformes aux cas calculés. Le module Lean cité comprend des cas fixés, un énoncé sur les listes d’opposants et, séparément, une équivalence générale pour `programNashCheck` ; ce carnet n’exécute aucune preuve Lean.


## Exercice 3 - Famille etendue aux budgets

On etend la famille temoin avec des variantes de budget : `basic_family_etendue = basic_family + [BoundedAgent(ProgramCode.MIRROR, 5), mirror_broke]`. **Question** : le profil de defection mutuelle `(defect_bot_bounded, defect_bot_bounded)` reste-t-il un equilibre relatif dans cette famille etendue ? Et la cooperation mutuelle des miroirs `(mirror_bot, mirror_bot)` devient-elle equilibre ? Completer le calcul des deux verdicts et interpreter pourquoi la reponse differe entre les deux profils.

Dans ce port, compléter le délégué C# correspondant ; les sondes utilisent aussi des familles variantes.


In [11]:
// Exercice ouvert : les deux profils dans la famille élargie et deux contrôles.
// Tester les profils défecteur/défecteur et miroir/miroir.
// Interpréter le gain d’une déviation contre le miroir.
var extendedFamily = basicFamily.Concat(new[] { new BoundedAgent(ProgramCode.Mirror, 5), mirrorBroke }).ToList();
Func<List<BoundedAgent>, BoundedAgent, BoundedAgent, (bool, bool)?> exercice3 = (family, d, m) => null;
var cases3 = new[]
{
    (extendedFamily, defectBot, mirrorBot),
    (new List<BoundedAgent> { cooperateBot }, cooperateBot, cooperateBot),
    (new List<BoundedAgent> { defectBot }, defectBot, defectBot)
};
var expected3 = cases3.Select(c => (ProgramNash(c.Item1, c.Item2, c.Item2), ProgramNash(c.Item1, c.Item3, c.Item3))).ToArray();
var replies3 = cases3.Select(c => exercice3(new List<BoundedAgent>(c.Item1), c.Item2, c.Item3)).ToArray();
if (replies3.All(r => r == null))
    Console.WriteLine("Exercice 3 à compléter.");
else
{
    if (replies3.Any(r => r == null))
        throw new Exception("Réponse partielle");
    for (int i = 0; i < cases3.Length; i++)
        if (replies3[i].Value != expected3[i])
            throw new Exception("Équilibre incorrect");
    Console.WriteLine("Exercice 3 : sondes réussies.");
}


Exercice 3 à compléter.


## À retenir

- **Le budget est public et paie la lecture** : la famille témoin vit à budget **0** pour le coopérateur et le défecteur — leur décision ne regarde ni l'adversaire ni l'historique — et le miroir seul porte **1** ; ce seul entier sépare le miroir qui réfléchit du miroir cassé.
- **La fonction `act` est totale** : aucune recherche de preuve, seulement l'inspection du code adverse et du budget ; à budget nul le miroir dévie, à budget positif il coopère sauf contre le code explicitement défecteur — et deux miroirs produisent une fois `(cooperate, cooperate)`.
- **Un rang fini pour l'ordre du dilemme** : sur le PD canonique (`T=5, R=3, P=1, S=0`), `payoffRank` aplatit l'ordre en **3** pour (C, C), **0** pour (C, D), **5** pour (D, C), **1** pour (D, D) — ces valeurs reprennent explicitement les gains canoniques et préservent leur ordre, sans établir de comparaison cardinale entre jeux.
- **Rejeu intégral des certificats** : le vérificateur C# indépendant reproduit les huit cas annoncés — **8/8** conformes, dont l'équivalence du rang fini vérifiée paire par paire sur **16** croisements.
- **Le partage des rôles** : le C# vérifie une matrice finie ; les énoncés Lean cités ont chacun leur portée propre, dont la quantification sur les listes d’opposants et l’équivalence générale séparée. Le carnet ne les exécute pas.

## Vérifiez votre compréhension

1. Quel budget le miroir porte-t-il dans la famille témoin, et que fait `act` pour un miroir à budget nul ?
   <details><summary>Réponse</summary>

   **Réponse.** Le miroir porte `budget=1` — un budget positif permet le test du code public. À budget nul, `act` renvoie `defect` immédiatement, sans inspection : la règle à budget nul prescrit immédiatement la défection.

   </details>

2. Quels rangs `payoffRank` assigne-t-il aux quatre issues, et quel ordre du dilemme ce classement encode-t-il ?
   <details><summary>Réponse</summary>

   **Réponse.** **3** pour (cooperate, cooperate), **0** pour (cooperate, defect), **5** pour (defect, cooperate), **1** pour (defect, defect) : l'ordre total T > R > P > S du PD canonique (`T=5, R=3, P=1, S=0`), aplati sur des entiers.

   </details>

3. Que dit la ligne finale `8/8 certificats conformes`, et que ne dit-elle pas ?
   <details><summary>Réponse</summary>

   **Réponse.** Que les huit cas annoncés ont le résultat attendu en C#, dont `payoffRank_le_iff` sur les **16** paires d’issues de `canonicalPD`. Ils ne démontrent pas des affirmations sur toutes les familles ou tous les budgets ; le module cité comporte notamment un énoncé sur les listes d’opposants et une équivalence générale distincte.

   </details>

4. Contre quel membre de la famille le coopérateur subit-il `(cooperate, defect)`, et pourquoi ne s'adapte-t-il pas ?
   <details><summary>Réponse</summary>

   **Réponse.** Contre `DEFECT_BOT b=0` : la sortie rend `('cooperate', 'defect')` pour cette paire. Le coopérateur coopère sans exception ni mémoire — il subit, il ne s'adapte pas.

   </details>

## Pour aller plus loin

**Carnet suivant.** [Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python](Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python.ipynb) - le coût du raisonnement borné, prolongement direct du budget explicite.

**Carnets voisins.** Pivot conceptuel : [Dynamics-04-Open-Source-Game-Theory-Python.ipynb](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) ; compagnon Lean : [Dynamics-05-Agents-Bornes-Lean.ipynb](Dynamics-05-Agents-Bornes-Lean.ipynb) ; module Lean (FR) : [game_theory_lean/ProgramGames/Bounded.lean](../game_theory_lean/ProgramGames/Bounded.lean) - miroir anglais : [Bounded_en.lean](../game_theory_lean/ProgramGames/Bounded_en.lean) ; noyau fonctionnel : [game_theory_lean/ProgramGames/Basic.lean](../game_theory_lean/ProgramGames/Basic.lean) ; lake complet : [game_theory_lean/README.md](../game_theory_lean/README.md).

**Lectures.** Barasz et al. (2014) et Critch (2016) - le modèle structurel de l'agent-programme à code public et budget de raisonnement fini que ce compagnon reproduit indépendamment depuis zéro.

## Navigation et suite

- Pivot conceptuel : [`Dynamics-04-Open-Source-Game-Theory-Python.ipynb`](Dynamics-04-Open-Source-Game-Theory-Python.ipynb)
- Module Lean (FR) : [`game_theory_lean/ProgramGames/Bounded.lean`](../game_theory_lean/ProgramGames/Bounded.lean) - miroir anglais : [`Bounded_en.lean`](../game_theory_lean/ProgramGames/Bounded_en.lean)
- Noyau fonctionnel : [`game_theory_lean/ProgramGames/Basic.lean`](../game_theory_lean/ProgramGames/Basic.lean)
- Lake complet : [`game_theory_lean/README.md`](../game_theory_lean/README.md)

Ce compagnon couvre le versant **calcul fini** du modele borne ; le versant formel vit dans le lake : les huit lignes ci-dessus ne sont pas huit théorèmes universels. Les exercices 1 a 3 restent a completer.
